# Lab 9: Experiments, Bayes learning, and bandits

The question for the night: when does a difference in the data mean that the change caused it, and how should beliefs and actions change as data arrive? We start with a difference too big to be noise, then run an A/B test, then let a Beta posterior and a bandit learn from the same visitors. We end with a way to judge a new rule using old data.

Sections:

1. Rayleigh's argon: a difference too big to be noise
2. A/B testing: SimCity, a two-proportion test, sample size
3. Peeking and multiple testing (with a computing corner on vectorized simulation and seeds)
4. Bayes learning: the same test with Beta posteriors (with a computing corner on map-reduce)
5. The death penalty in Florida: Simpson's paradox
6. Bandits: Thompson sampling, the A/B test that learns while it runs
7. Importance sampling: judging a new rule with old data
8. Why randomize: Washington's plots
9. Russian election observers
10. Gujarat pollution audits (simulated)
11. Exit problem, then At home

How to work: run the cells top to bottom. **Predict** items: write your guess before you run the cell. **Your turn** cells run as given: change the line marked `# change this`. **Check yourself** answers are hidden under a click. **Lab exercises** (Exercises 9.1 to 9.6) are the cells you fill in and submit on Canvas: 9.1 to 9.4 in class, 9.5 and 9.6 at home. Every random step uses a fixed seed, so your numbers match the ones quoted here.


## Route for this notebook

**Three outcomes.** After this lab you should be able to:

1. Test and interval the difference of two rates, and say how many visitors a test needs.
2. Update a Beta belief by adding counts, and let a bandit act while it learns.
3. Compute an importance weight (new probability divided by old probability) and say when it fails.

**Core sections:** all of them, with the plan for class time (160 minutes): 1 Rayleigh (12), 2 A/B testing (25), 3 peeking and multiple testing (10), 4 Bayes learning (20), break (10), 5 Simpson's paradox (10), 6 bandits (20), 7 importance sampling (15), 8 why randomize (5), 9 Russian observers (15), 10 Gujarat audits (8), exit problem (10). Sections we do not finish in class are completed from the notebook.

**Required:** the six graded lab exercises, 9.1 to 9.6 (9.1 to 9.4 in class, 9.5 and 9.6 at home).

**Due:** before class on 11 November. Submit the notebook on Canvas.

Run All stops at the first empty exercise check; that is expected. Fill the exercises, then Restart and Run All before you submit.


## Online labs for this notebook

Open these in another tab. Each one lets you play with an idea from the night.

- [A/B Test Lab](https://vsokolov.org/labs/ab-test-lab.html): run a two-page test and watch the difference and its interval (Section 2).
- [Power and Sample Size Lab](https://vsokolov.org/labs/power-lab.html): see how many visitors a small lift needs (Section 2).
- [Confidence Interval Lab](https://vsokolov.org/labs/ci-coverage-lab.html): see what "95%" means over many repeats (Section 2).
- [Multiple Testing Lab](https://vsokolov.org/labs/multiple-testing-lab.html): see false winners appear as the number of tests grows (Section 3).
- [Bandit vs A/B Lab](https://vsokolov.org/labs/bandit-lab.html): Thompson sampling against a fixed split (Section 6).
- [Rent Control Lab](https://vsokolov.org/labs/homeless-lab.html): an observational regression whose estimate moves with the specification, and a permutation test (Sections 5, 8 and 9).


In [ ]:
import time
from hashlib import sha256

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.rcParams["figure.figsize"] = (5, 3.5)

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    try:
        s = f"{round(float(value), ndigits):.{ndigits}f}"
    except (TypeError, ValueError):
        raise AssertionError(f"{name} must be a single number, not {type(value).__name__}")
    if not np.isfinite(float(value)):
        raise AssertionError(f"{name} is not finite (nan or inf)")
    got = sha256(f"aii600-lab9|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} does not match the reference value (you got {s}); check the formula and the inputs"
    print("ok ", name)

def check_array(name, value, expected, shape, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    value = np.asarray(value, dtype=float)
    if value.shape != shape:
        raise AssertionError(f"{name} must have shape {shape}, but has shape {value.shape}")
    if not np.all(np.isfinite(value)):
        raise AssertionError(f"{name} contains nan or inf")
    s = ",".join(f"{round(float(v), ndigits):.{ndigits}f}" for v in value.ravel())
    got = sha256(f"aii600-lab9|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} has the right shape but the values do not match the reference; check each update"
    print("ok ", name)

# This lab needs no data files. The data are typed in below or simulated, and every simulation says so.


## 1. Rayleigh's argon: a difference too big to be noise

Lord Rayleigh weighed the same volume of nitrogen from two sources. One sample came from air, after he removed the oxygen. The other came from a chemical reaction (decomposing nitrous oxide and nitric oxide). The numbers below are densities (grams per liter). One air measurement is missing in the book's table, so air has 7 values and chemical decomposition has 8.

The question: is the gap between the two means just measurement noise? The two-sample $t$-statistic answers it. It measures the gap in units of its standard error. The two groups have very different spreads (see the printed standard deviations), so we use Welch's version, which does not assume a common spread:

$$ t = \frac{\bar x_1 - \bar x_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}. $$

Here $\bar x$ is a mean, $s$ a standard deviation, and $n$ a sample size. A large $|t|$ (say above 2) means the gap is large relative to what the null model (no real gap, only noise) would typically produce.


In [ ]:
air = np.array([2.31017, 2.30986, 2.31010, 2.31001, 2.31024, 2.31010, 2.31028])
decomp = np.array([2.30143, 2.29890, 2.29816, 2.30182, 2.29869, 2.29940, 2.29849, 2.29889])
print(f"air   : n={len(air)}, mean={air.mean():.5f}, sd={air.std(ddof=1):.6f}")
print(f"decomp: n={len(decomp)}, mean={decomp.mean():.5f}, sd={decomp.std(ddof=1):.6f}")

n1, n2 = len(air), len(decomp)
se_welch = np.sqrt(air.var(ddof=1) / n1 + decomp.var(ddof=1) / n2)
t_by_hand = (air.mean() - decomp.mean()) / se_welch
res = stats.ttest_ind(air, decomp, equal_var=False)            # Welch
res_pooled = stats.ttest_ind(air, decomp, equal_var=True)      # assumes one common spread
print(f"Welch t by hand = {t_by_hand:.1f}, scipy Welch t = {res.statistic:.1f}, p-value = {res.pvalue:.1e}")
print(f"(the pooled version, which assumes a common spread, gives t = {res_pooled.statistic:.1f})")


**Reading.** The means are 2.31 and 2.30, which look close. But the spread inside each group is tiny (standard deviations about 0.0001 and 0.0014), so the gap of about 0.011 is huge compared with the noise. Welch's $t$-statistic is about 21.7 and the p-value is about 8e-08. The pooled version gives 20.2, also far above 2, so the conclusion does not depend on the choice. Rayleigh took the gap seriously, and it led to the discovery of argon: the gas from air had a little of a heavier gas in it. The same idea, the gap divided by its standard error, returns for SimCity.


In [ ]:
# Your turn: keep only a fraction of the real gap and watch t fall.
# Target: find a value of frac where t drops below 2.
frac = 1.0   # change this: 1.0 = the real data, 0.0 = no gap at all
shifted = decomp - (1 - frac) * (decomp.mean() - air.mean())   # slides decomp toward air
t_new = stats.ttest_ind(air, shifted, equal_var=False).statistic
print(f"frac = {frac}: gap = {air.mean() - shifted.mean():.5f}, t = {t_new:.1f}")


**Check yourself.**

1. The two means differ by only about 0.5%. Why is the $t$-statistic so large?
2. If you doubled the number of measurements in each group and the gap and spread stayed the same, would $t$ go up or down?

<details><summary>Answer</summary>

1. The standard error is tiny, because the measurements inside each group are very close to each other. $t$ compares the gap with the noise, not with the size of the numbers.
2. Up, by about a factor of $\sqrt 2$. The standard error shrinks like $1/\sqrt n$.

</details>


## 2. A/B testing: SimCity

Electronic Arts tested a pink banner against a black one on the pre-order page for SimCity. We use the numbers from the book for one week:

- pink banner: 40% of 500 visitors bought
- black banner: 30% of 550 visitors bought

Let $p_1$ be the true purchase rate with the pink banner and $p_2$ with the black one. We see only the sample rates $\hat p_1 = 0.40$ and $\hat p_2 = 0.30$. We want to know whether $p_1 - p_2$ is really above zero.

**One rate.** If $n$ visitors each buy with probability $p$, the sample rate $\hat p$ has standard error $\sqrt{p(1-p)/n}$ (we plug in $\hat p$ for $p$).

**The difference of two rates.** The two groups are different visitors, so the two errors are independent and their variances add:

$$ \mathrm{SE}_{\text{diff}} = \sqrt{\frac{\hat p_1(1-\hat p_1)}{n_1} + \frac{\hat p_2(1-\hat p_2)}{n_2}}, \qquad \text{95\% interval: } (\hat p_1 - \hat p_2) \pm 1.96\, \mathrm{SE}_{\text{diff}}. $$

**The z-test.** The null hypothesis is that both pages have the same rate. Then we may pool all visitors to estimate that one rate, $\hat p_0 = (\text{all buys})/(\text{all visitors})$, and

$$ z = \frac{\hat p_1 - \hat p_2}{\sqrt{\hat p_0 (1-\hat p_0)(1/n_1 + 1/n_2)}}. $$

A $|z|$ above 1.96 rejects the null at 5%. The two-sided p-value is $2\,(1 - \Phi(|z|))$, where $\Phi$ is the standard normal distribution function.

**Online labs.** Open the [A/B Test Lab](https://vsokolov.org/labs/ab-test-lab.html) next to this section and run the SimCity test (40% against 30%) with 500 and 550 visitors. Then open the [Confidence Interval Lab](https://vsokolov.org/labs/ci-coverage-lab.html) and repeat the experiment many times: about 95 of 100 intervals should contain the true value, and that is what "95%" means.

**Why not compare two separate intervals?** The book checks whether two 80% intervals overlap. That is not a test of the difference: the standard error of the difference is $\sqrt{\mathrm{SE}_1^2 + \mathrm{SE}_2^2}$, which is smaller than $\mathrm{SE}_1 + \mathrm{SE}_2$, so two intervals can overlap even when the difference is clearly real.


In [ ]:
# SimCity: one week of visitors (from the book: pink 40% of 500, black 30% of 550)
n_pink, buys_pink = 500, 200
n_black, buys_black = 550, 165
rate_pink = buys_pink / n_pink
rate_black = buys_black / n_black
print(f"pink : {buys_pink} buys of {n_pink} visitors, rate {rate_pink:.3f}")
print(f"black: {buys_black} buys of {n_black} visitors, rate {rate_black:.3f}")


In [ ]:
zc = stats.norm.ppf(0.975)                      # 1.96
se1 = np.sqrt(rate_pink * (1 - rate_pink) / n_pink)
se2 = np.sqrt(rate_black * (1 - rate_black) / n_black)
se_diff = np.sqrt(se1**2 + se2**2)
diff = rate_pink - rate_black
print(f"difference = {diff:.3f}, SE of difference = {se_diff:.4f}")
print(f"95% interval for the difference: ({diff - zc * se_diff:.3f}, {diff + zc * se_diff:.3f})")

p0 = (buys_pink + buys_black) / (n_pink + n_black)
z_stat = diff / np.sqrt(p0 * (1 - p0) * (1 / n_pink + 1 / n_black))
print(f"pooled rate = {p0:.3f}, z = {z_stat:.2f}, p-value = {2 * (1 - stats.norm.cdf(abs(z_stat))):.1e}")
print(f"separate 95% intervals: pink ({rate_pink - zc * se1:.3f}, {rate_pink + zc * se1:.3f}), black ({rate_black - zc * se2:.3f}, {rate_black + zc * se2:.3f})")
print(f"SE1 + SE2 = {se1 + se2:.4f}, but SE of the difference = {se_diff:.4f}")


**Reading.** The pink banner is ahead by 10 points. The 95% interval for the difference is (0.042, 0.158), and it stays well above zero. The $z$-statistic is 3.40, so the p-value is about 0.0007: this is very unlikely if the pages were equal. The last line shows the point about overlap: $\mathrm{SE}_1 + \mathrm{SE}_2$ is larger than the true standard error of the difference.

Exercise 9.1 below asks for the interval.


### Exercise 9.1 (In class)

Task: compute the 95% interval for the difference of the two SimCity purchase rates (pink minus black). Use the separate (unpooled) standard errors from the formula above and $z = 1.96$ (use `stats.norm.ppf(0.975)`).

Inputs: `n_pink`, `buys_pink`, `n_black`, `buys_black`.

Define: `diff_lo` and `diff_hi`, two numbers (rates, so about 0.04 means 4 points).


In [ ]:
# Exercise 9.1: define diff_lo, diff_hi


In [ ]:
check("diff_lo", diff_lo, "22d17664c8d21c05")
check("diff_hi", diff_hi, "3ac43e225227a1c6")
print("ok  Exercise 9.1")


The next step is the other direction: how many visitors do we need before the test, so that a small real lift would be found?

**Sample size per arm.** This is a normal approximation. To detect a true lift from $p_1$ to $p_2$ with 5% significance (two-sided) and 80% power (the test finds the lift 80% of the time when it is real):

$$ n = \frac{(z_{0.975} + z_{0.80})^2 \,\big[p_1(1-p_1) + p_2(1-p_2)\big]}{(p_2 - p_1)^2}. $$

Here $z_{0.975}$ and $z_{0.80}$ are normal quantiles (about 1.96 and 0.84), and $n$ is the number of visitors in each arm. In code, use `stats.norm.ppf(0.975)` and `stats.norm.ppf(0.80)`, not the rounded numbers, and round up with `np.ceil` to an integer.

Open the [Power and Sample Size Lab](https://vsokolov.org/labs/power-lab.html) and set the lift from 30% to 32%: the sample size it shows should be close to the one you compute in Exercise 9.2.


**Predict.** If the lift is cut in half, from 5 points to 2.5 points, will the sample size per arm double, triple, or quadruple? Write your answer, then run the cell below with `lift = 0.025`. The answer is under the next cell.


In [ ]:
# Your turn: the formula as a function. The cell runs with a 5-point lift from 30%.
# Target: change `lift`. What happens to n when the lift is half as big? (Use stats.norm.ppf for the z values.)
def n_per_arm(p_start, lift, power=0.80):
    p_end = p_start + lift
    z_sum = stats.norm.ppf(0.975) + stats.norm.ppf(power)
    return int(np.ceil(z_sum**2 * (p_start * (1 - p_start) + p_end * (1 - p_end)) / lift**2))

lift = 0.05   # change this
print(f"lift {lift:.3f} from 30%: n = {n_per_arm(0.30, lift)} visitors per arm")


<details><summary>Answer to the Predict</summary>

About four times as many: halving the lift roughly quadruples the required sample size, because $n$ grows like $1/\text{lift}^2$ (the variance terms change slightly, so it is not exactly four).

</details>


### Exercise 9.2 (In class)

Task: find the number of visitors per arm needed to detect a lift from a 30% purchase rate to 32% (a 2-point lift), with 5% significance (two-sided) and 80% power, using the formula above. Use `stats.norm.ppf(0.975)` and `stats.norm.ppf(0.80)` (not 1.96 and 0.84), and round up with `np.ceil` to an `int`.

Inputs: $p_1 = 0.30$, $p_2 = 0.32$.

Define: `n_arm`, an integer (round up).


In [ ]:
# Exercise 9.2: define n_arm


In [ ]:
check("n_arm", n_arm, "5c67b94d7c279872")
print("ok  Exercise 9.2")


**Check yourself.**

1. A friend says the two 95% intervals for pink and black overlap a little, so the test is not significant. Is that right?
2. Why does a 2-point lift need many more visitors than a 10-point lift?

<details><summary>Answer</summary>

1. No. Overlapping intervals do not decide the test. The interval for the difference uses $\sqrt{\mathrm{SE}_1^2 + \mathrm{SE}_2^2}$, which is smaller than $\mathrm{SE}_1 + \mathrm{SE}_2$. Test the difference directly.
2. $n$ grows approximately like $1/\text{lift}^2$ (the variance terms also change a little). A lift 5 times smaller needs about 25 times as many visitors.

</details>


## 3. Peeking and multiple testing

An A/B test is valid when you fix the sample size first, run to the end, and test once. Two common habits break this.

**Peeking.** You look at the test after every 100 visitors per arm and stop as soon as $|z| > 1.96$. Each look is a chance for noise to cross the line.

**Many variants.** You test 20 banners against the control, and call a winner if any one gives $|z| > 1.96$.

Open the [Multiple Testing Lab](https://vsokolov.org/labs/multiple-testing-lab.html) next to this section and raise the number of tests to 20: count how many look significant when nothing is real.

We check both with simulated A/A tests: both arms have the same true rate 0.30, so any "winner" is a false positive. A fair test gives a false positive 5% of the time. All visitors below are simulated.

### Computing corner: vectorized simulation

We need 1,000 experiments. A Python loop runs them one at a time. A vectorized version draws all of them in one array operation. The array has shape (1000 experiments, 20 looks), and each entry is the number of buyers among 100 new visitors. Cumulative sums along the looks give the running totals.


In [ ]:
def peeking_vectorized(n_exp=1000, looks=20, block=100, rate=0.30, seed=11):
    rng = np.random.default_rng(seed)
    buys1 = rng.binomial(block, rate, size=(n_exp, looks)).cumsum(axis=1)   # arm A running buys
    buys2 = rng.binomial(block, rate, size=(n_exp, looks)).cumsum(axis=1)   # arm B running buys
    n_seen = block * np.arange(1, looks + 1)                                # visitors per arm so far
    p_hat = (buys1 + buys2) / (2 * n_seen)
    z = (buys1 - buys2) / n_seen / np.sqrt(p_hat * (1 - p_hat) * 2 / n_seen)
    return z

def peeking_loop(n_exp=1000, looks=20, block=100, rate=0.30, seed=11):
    """The beginner's version: two loops, one random draw at a time. Returns the share of false positives."""
    rng = np.random.default_rng(seed)
    n_false = 0
    for e in range(n_exp):
        b1 = b2 = 0
        for look in range(1, looks + 1):
            b1 += rng.binomial(block, rate)
            b2 += rng.binomial(block, rate)
            n_seen = block * look
            p_hat = (b1 + b2) / (2 * n_seen)
            z = (b1 - b2) / n_seen / np.sqrt(p_hat * (1 - p_hat) * 2 / n_seen)
            if abs(z) > 1.96:
                n_false += 1
                break                                    # stop at the first look that crosses
    return n_false / n_exp

t0 = time.perf_counter(); z_vec = peeking_vectorized(); t_vec = time.perf_counter() - t0
t0 = time.perf_counter(); fp_loop = peeking_loop(); t_loop = time.perf_counter() - t0
print(f"vectorized: {t_vec * 1000:.1f} ms, double loop: {t_loop * 1000:.1f} ms, the loop is {t_loop / t_vec:.0f} times slower")

fp_end = (np.abs(z_vec[:, -1]) > 1.96).mean()            # test once, at 2,000 visitors per arm
fp_peek = (np.abs(z_vec) > 1.96).any(axis=1).mean()       # stop at the first look that crosses
print(f"false positives, testing once at the end : {fp_end:.3f}")
print(f"false positives, peeking after every 100 : {fp_peek:.3f}")
print(f"false positives, peeking, double-loop version : {fp_loop:.3f}")


**Reading.** Testing once at the end gives a false-positive rate of 0.051, close to the promised 0.05. Peeking after every 100 visitors per arm (20 looks) gives 0.234, about 4.6 times too many. The cell also prints the time of both versions. The double loop, which draws one number at a time, is several times slower than the array version (the exact times change from run to run, and the gap is larger for bigger simulations). The loop gives a false-positive rate near the same value (it uses other random numbers). Rule: write simulations as array operations, and loop only over things that depend on the previous step, such as the visitors in a bandit.

### Many variants

Now 20 variants against one control, 1,000 visitors per arm, all with the same true rate 0.30. Count how many variants look significant in each experiment.


In [ ]:
def many_variants(n_exp=1000, k=20, n=1000, rate=0.30, seed=12):
    rng = np.random.default_rng(seed)
    control = rng.binomial(n, rate, size=(n_exp, 1))
    variants = rng.binomial(n, rate, size=(n_exp, k))
    p_hat = (control + variants) / (2 * n)
    z = (variants - control) / n / np.sqrt(p_hat * (1 - p_hat) * 2 / n)
    return 2 * (1 - stats.norm.cdf(np.abs(z)))          # p-values, shape (n_exp, k)

pv = many_variants()
sig = (pv < 0.05).sum(axis=1)
print(f"average number of false winners per experiment: {sig.mean():.2f}")
print(f"share of experiments with at least one false winner: {(sig >= 1).mean():.3f}  (theory: {1 - 0.95**20:.3f})")
print(f"Bonferroni (cutoff 0.05/20 = {0.05 / 20}): share with at least one false winner: {(pv < 0.05 / 20).any(axis=1).mean():.3f}")


**Reading.** With 20 variants, an experiment has on average 0.94 false winners, and 0.413 of experiments show at least one. For 20 independent tests the theory says $1 - 0.95^{20} = 0.642$. The simulation gives less because the 20 variants share one control: if the control is unlucky, many variants look significant together, so the false winners tend to appear together. The average number of false winners is still about 1. The Bonferroni rule divides the cutoff by the number of tests, here $0.05/20 = 0.0025$. It brings the chance of any false winner back to about 0.025. Bonferroni guarantees at most 0.05, and the result is lower here because the tests share the control. The price is that a real lift now needs a larger $z$ to be found.

### Computing corner: seeds

Every simulation above starts with `np.random.default_rng(seed)`. The same seed gives the same numbers, which makes a result reproducible. Two different experiments should not share one stream by accident, so give each its own seed. `SeedSequence(...).spawn(k)` makes `k` independent streams from one master seed.


In [ ]:
print("same seed, run twice:", np.random.default_rng(1).random(3).round(3), np.random.default_rng(1).random(3).round(3))
print("different seed      :", np.random.default_rng(2).random(3).round(3))

streams = [np.random.default_rng(s) for s in np.random.SeedSequence(2026).spawn(3)]
print("three spawned streams, first draw each:", [round(g.random(), 3) for g in streams])
print("spawn again from the same master seed :", [round(np.random.default_rng(s).random(), 3) for s in np.random.SeedSequence(2026).spawn(3)])


**Check yourself.**

1. A manager checks the dashboard daily for a month and ships the first variant that is significant at 5%. What is wrong?
2. Why does the Bonferroni rule lose power?

<details><summary>Answer</summary>

1. Each daily look is another test, so the false-positive rate is much higher than 5%. Fix the sample size in advance (or use a method built for repeated looks).
2. A smaller cutoff means a larger $z$ is needed, so real but small lifts are missed more often.

</details>


## 4. Bayes learning: the same test with Beta posteriors

The z-test gave an estimate, an interval, a statistic and a p-value. A Bayesian analysis gives a whole distribution for each purchase rate, and it can say directly how likely it is that pink is better.

**The Beta distribution.** A Beta($a$, $b$) distribution lives on $[0, 1]$, so it fits a rate $p$. Its mean is $a/(a+b)$. Beta(1, 1) is flat: every rate is equally likely before we see data. This is the **prior** (our belief before the data).

**The update.** After $n$ visitors with $s$ buys (successes) and $n - s$ non-buys, the **posterior** (our belief after the data) is

$$ \text{Beta}(a, b) \;\longrightarrow\; \text{Beta}(a + s,\; b + n - s). $$

The update is adding counts: prior counts plus data counts. Starting from Beta(1, 1), pink (200 buys of 500) gives Beta(201, 301) and black (165 buys of 550) gives Beta(166, 386).

To find $P(p_1 > p_2)$ (the probability that pink's true rate is higher), draw many values from each posterior and count how often the pink draw is larger.


In [ ]:
a0, b0 = 1, 1                                   # the flat prior, Beta(1, 1)
post_pink = (a0 + buys_pink, b0 + n_pink - buys_pink)
post_black = (a0 + buys_black, b0 + n_black - buys_black)
print("posterior pink : Beta", post_pink, " mean", round(post_pink[0] / sum(post_pink), 3))
print("posterior black: Beta", post_black, " mean", round(post_black[0] / sum(post_black), 3))

rng_b = np.random.default_rng(3)
draw_pink = rng_b.beta(*post_pink, size=100_000)
draw_black = rng_b.beta(*post_black, size=100_000)
print("P(pink > black) by sampling:", (draw_pink > draw_black).mean())
print("95% credible interval for pink :", np.percentile(draw_pink, [2.5, 97.5]).round(3))
print("95% credible interval for black:", np.percentile(draw_black, [2.5, 97.5]).round(3))

grid = np.linspace(0, 1, 500)
plt.plot(grid, stats.beta.pdf(grid, a0, b0), color="gray", linestyle=":", label="prior Beta(1, 1)")
plt.plot(grid, stats.beta.pdf(grid, *post_pink), color="C3", label="pink posterior")
plt.plot(grid, stats.beta.pdf(grid, *post_black), color="C0", linestyle="--", label="black posterior")
plt.xlim(0.15, 0.55)
plt.xlabel("purchase rate")
plt.ylabel("density")
plt.legend()
plt.show()


**Reading.** The posterior means are 0.400 for pink and 0.301 for black, close to the sample rates. The 95% credible intervals (the range holding 95% of the posterior) do not overlap and the two curves are far apart. The probability that pink is better is 0.99966 by sampling. The prior is flat, so the data decide almost everything.

**Predict.** Suppose we had only 50 visitors per page with the same rates (pink 20 of 50, black 15 of 50). Will the two posteriors be wider or narrower? Will P(pink > black) be higher or lower than above? Write your answer, then run the cell. The answer is in the reading after the output.


### Exercise 9.3 (In class)

Task: find the probability that pink's true rate is higher than black's, using the two SimCity posteriors from a flat Beta(1, 1) prior. Draw at least 100,000 values from each posterior with your own `np.random.default_rng(seed)` and count how often the pink draw is larger.

Inputs: `buys_pink`, `n_pink`, `buys_black`, `n_black`.

Define: `p_pink_better`, a number between 0 and 1. The check allows a small tolerance.


In [ ]:
# Exercise 9.3: define p_pink_better


In [ ]:
assert p_pink_better is not None, "replace None for p_pink_better"
assert abs(p_pink_better - 0.999658) < 0.002, "p_pink_better is not within 0.002 of the target"
print("ok  Exercise 9.3")


In [ ]:
post_pink_50 = (a0 + 20, b0 + 30)
post_black_50 = (a0 + 15, b0 + 35)
d_pink = rng_b.beta(*post_pink_50, size=100_000)
d_black = rng_b.beta(*post_black_50, size=100_000)
print("50 visitors per page: P(pink > black) =", (d_pink > d_black).mean())
print("posterior sd, pink: 500 visitors", round(stats.beta.std(*post_pink), 4), " 50 visitors", round(stats.beta.std(*post_pink_50), 4))


**Reading.** With 50 visitors the posteriors are wider (standard deviation 0.067 against 0.022 for pink) and P(pink > black) drops to 0.85. The same rates with less data give less certainty. This is the Bayesian version of "a smaller sample gives a larger standard error".


**Your turn.** A strong prior: before the test, we believe the rate is 30%, and the belief is worth 300 pretend visitors, so Beta(90, 210). The cell shows the posterior mean for pink at several sample sizes, with the true rate 0.40. Target: find the number of visitors at which the posterior mean reaches 0.35 (halfway between the prior's 0.30 and the data's 0.40). Change `strength` to see how the answer moves.


In [ ]:
strength = 300          # change this: the prior is worth this many pretend visitors
prior_a, prior_b = 0.30 * strength, 0.70 * strength
for n_vis in [50, 100, 200, 300, 500, 1000, 3000]:
    buys = 0.40 * n_vis                       # data at exactly 40%
    post_mean = (prior_a + buys) / (prior_a + prior_b + n_vis)
    print(f"{n_vis:5d} visitors: posterior mean {post_mean:.3f}")


**Reading.** With strength 300 the posterior mean passes 0.35 at 300 visitors, the point where the data count as much as the prior. Before that, the prior wins. After that, the data win. A stronger prior needs more data.

The update is a sum of counts. This has two consequences.

1. **One visitor at a time.** Adding 1 to $a$ for a buy and 1 to $b$ for a non-buy gives the same posterior as processing everyone at once. Bandits use this.
2. **Many machines.** Each machine can count its own buys and visitors, and the counts can be added.

### Computing corner: map-reduce on sufficient statistics

A **sufficient statistic** is a small summary that holds everything the data say about the parameter. Here it is (buys, visitors). Below, the 500 pink visitors' log is split over 4 "servers" (four arrays).

- **Map:** each server counts its own buys and visitors.
- **Reduce:** add the four pairs of counts.

The posterior from the shards should be identical to the posterior from the full log.


In [ ]:
rng_s = np.random.default_rng(4)
pink_log = np.r_[np.ones(buys_pink), np.zeros(n_pink - buys_pink)]      # 200 buys, 300 non-buys
rng_s.shuffle(pink_log)
servers = np.array_split(pink_log, 4)

mapped = [(int(s.sum()), len(s)) for s in servers]                       # map: (buys, visitors) per server
print("per-server (buys, visitors):", mapped)
buys_total = sum(m[0] for m in mapped)                                   # reduce: add the counts
n_total = sum(m[1] for m in mapped)
print("reduced posterior : Beta", (a0 + buys_total, b0 + n_total - buys_total))
print("full-log posterior: Beta", (a0 + int(pink_log.sum()), b0 + len(pink_log) - int(pink_log.sum())))


**Reading.** Each server sends back two numbers instead of its whole log. The reduced posterior, Beta(201, 301), equals the one from the full log. The data stay where they are, and only counts travel. On 2 December the same pattern scales up: each GPU computes a gradient on its shard, and the gradients are added.


**Check yourself.**

1. A Beta(1, 1) prior and then 12 buys in 40 visitors. What is the posterior, and its mean?
2. Why can a Beta posterior be updated one visitor at a time?

<details><summary>Answer</summary>

1. Beta(1 + 12, 1 + 28) = Beta(13, 29). Its mean is 13/42 = 0.310.
2. The update only adds counts, and addition does not depend on the order or on how the data are grouped.

</details>


## 5. The death penalty in Florida: Simpson's paradox

**The data.** Michael Radelet (1981, *American Sociological Review*) studied 326 defendants charged with murder in 20 Florida counties in 1976 and 1977. For each defendant the table gives the defendant's race, the victim's race, and the outcome: a death sentence or not. The book's table (chapter 16) is used here. The race labels in the book's text are reversed; the data are as below, with explicit names.

**What the data can and cannot show.** They are counts of cases in two years in one state, and they say nothing about the facts of each case (the severity of the crime, prior record, the evidence). They are observational: nobody can randomly assign the race of a defendant or a victim. So they can show a pattern; they cannot show why a particular sentence was given, and they cannot show that any single decision was fair or unfair. We use them to see how a table can mislead when a third variable is hidden. On 18 November the same table returns for Bayesian model comparison.


In [ ]:
cases = pd.DataFrame({
    "defendant": ["white", "white", "Black", "Black"],
    "victim": ["white", "Black", "white", "Black"],
    "death": [19, 0, 11, 6],
    "no_death": [132, 9, 52, 97],
})
cases["cases"] = cases.death + cases.no_death
cases["death_rate"] = (cases.death / cases.cases).round(3)
print(cases.to_string(index=False))
print("total cases:", cases.cases.sum())

by_def = cases.groupby("defendant")[["death", "cases"]].sum()
by_def["death_rate"] = (by_def.death / by_def.cases).round(3)
print()
print("Overall, by defendant's race:")
print(by_def)


**Predict.** The overall table above: which defendants received the death penalty more often, white or Black? Now guess: inside the group of cases with a white victim, will the order stay the same or reverse? Write your answer, then run the next cell.


In [ ]:
print("Within each victim group (death rate by defendant's race):")
within = cases.pivot(index="victim", columns="defendant", values="death_rate")
print(within)

share_white_def_white_victim = 151 / 160
share_black_def_white_victim = 63 / 166
print()
print(f"white defendants whose victim was white: {share_white_def_white_victim:.1%}")
print(f"Black defendants whose victim was white: {share_black_def_white_victim:.1%}")

by_vic = cases.groupby("victim")[["death", "cases"]].sum()
by_vic["death_rate"] = (by_vic.death / by_vic.cases).round(3)
print()
print("By victim's race alone:")
print(by_vic)


**Reading.** Overall, white defendants got the death penalty more often: 19 of 160, 11.9%, against 17 of 166, 10.2% for Black defendants. But inside each victim group the order reverses. With a white victim, the rates are 12.6% (white defendants) and 17.5% (Black defendants). With a Black victim, they are 0% and 5.8%. This is **Simpson's paradox**: a comparison of the whole can point the opposite way from the comparison inside every group.

The reason is the victim's race, the **confounder** (a third variable tied to both the group and the outcome). Most cases have a defendant and victim of the same race. So 94% of white defendants (151 of 160) had a white victim, against 38% of Black defendants (63 of 166). And cases with a white victim received the death penalty far more often (14.0% against 5.4% for a Black victim). The overall rate for white defendants is a mix that leans on the high-rate group.

**Check yourself.** Which comparison, the overall one or the within-group one, answers the question "does the defendant's race matter, for cases with the same victim race"? And can the within-group comparison settle it?

<details><summary>Answer</summary>

The within-group comparison, because it compares cases with the same victim race. It cannot settle it: other differences between cases (the facts of the crime, the evidence, prior record) are not in the table, and these data are not randomized. We can only say that the overall comparison is a poor guide, and that the answer depends on what we hold fixed.

</details>

The table has small groups (0 of 9 for white defendants with a Black victim), so we do not claim that any single difference is real. The point is only that the overall table and the group tables can disagree. The strongest pattern in the table has nothing to do with the defendant: death sentences were more common when the victim was white (30 of 214, 14.0%) than when the victim was Black (6 of 112, 5.4%). This is a pattern in the counts, not a causal claim.

Race cannot be randomized. That is why we need the idea of the next sections: randomizing balances the other differences between groups.


## 6. Bandits: Thompson sampling, the A/B test that learns while it runs

An A/B test learns, then acts: it splits traffic 50/50 for the whole test, and only afterward picks a winner. During the test, half the visitors see the worse page. A **bandit** learns while it acts: it sends more visitors to the page that looks better, and keeps learning from the rest. This is the problem of **explore and exploit**: try other options to learn (explore), or use the best one so far to earn (exploit).

**Thompson sampling.** Keep a Beta posterior for each page. For each new visitor:

1. Draw one value from each page's posterior.
2. Show the page whose draw is larger.
3. See whether the visitor buys, and add that to that page's Beta counts.

A page that looks better gets drawn high more often, so it gets more traffic. A page that is still uncertain has a wide posterior, so it sometimes wins the draw and gets tried again.

Open the [Bandit vs A/B Lab](https://vsokolov.org/labs/bandit-lab.html) next to this section and run Thompson sampling against the 50/50 split with rates 0.40 and 0.30: watch the share of traffic shift to the better page.

We compare Thompson sampling with the 50/50 test in simulation. The true rates are pink 0.40 and black 0.30 (simulated; the algorithm does not know them). We run 200 independent experiments of 5,000 visitors at once, so the loop is over visitors and each step is an array operation over the 200 experiments.

**Regret** is the number of purchases lost against an oracle that always shows pink: for each visitor, (0.40 minus the true rate of the page shown), summed over visitors.


**Predict.** Does the Thompson bandit ever stop sending visitors to the worse page? After 5,000 visitors, what share of traffic do you expect on pink: 60%, 90%, 100%? Write your answer, then run the cell after this one.


In [ ]:
def run_policy(policy, T=5000, R=200, seed=0, rates=(0.40, 0.30), eps=0.1, alpha=None):
    """Return the arm shown at each visitor, shape (R, T). Arm 0 = pink, arm 1 = black."""
    rng = np.random.default_rng(seed)
    rates = np.array(rates)
    rows = np.arange(R)
    a = np.ones((R, 2)); b = np.ones((R, 2))          # Beta counts (Thompson)
    Q = np.zeros((R, 2)); cnt = np.zeros((R, 2))      # running estimates (epsilon-greedy)
    arms = np.zeros((R, T), dtype=int)
    for t in range(T):
        if policy == "ab":
            arm = rng.integers(0, 2, R)
        elif policy == "thompson":
            arm = rng.beta(a, b).argmax(axis=1)
        else:  # epsilon-greedy
            greedy = (Q + 1e-9 * rng.random((R, 2))).argmax(axis=1)
            arm = np.where(rng.random(R) < eps, rng.integers(0, 2, R), greedy)
        r = (rng.random(R) < rates[arm]).astype(float)
        a[rows, arm] += r
        b[rows, arm] += 1 - r
        cnt[rows, arm] += 1
        step = (1 / cnt[rows, arm]) if alpha is None else alpha
        Q[rows, arm] += step * (r - Q[rows, arm])     # Q <- Q + alpha (r - Q)
        arms[:, t] = arm
    return arms

true_rates = np.array([0.40, 0.30])
arms_ab = run_policy("ab", seed=60)
arms_ts = run_policy("thompson", seed=61)

def regret_curve(arms):
    return (0.40 - true_rates[arms]).cumsum(axis=1).mean(axis=0)       # average over the 200 experiments

def pink_share(arms, window=100):
    x = (arms == 0).mean(axis=0)
    return np.convolve(x, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(pink_share(arms_ab), color="C0", linestyle="--", label="50/50 test")
ax[0].plot(pink_share(arms_ts), color="C3", label="Thompson sampling")
ax[0].set_xlabel("visitor number"); ax[0].set_ylabel("share of visitors shown pink"); ax[0].legend()
ax[1].plot(regret_curve(arms_ab), color="C0", linestyle="--", label="50/50 test")
ax[1].plot(regret_curve(arms_ts), color="C3", label="Thompson sampling")
ax[1].set_xlabel("visitor number"); ax[1].set_ylabel("purchases lost (regret)"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"regret after 5,000 visitors: 50/50 test {regret_curve(arms_ab)[-1]:.0f}, Thompson {regret_curve(arms_ts)[-1]:.0f}")
print(f"share shown pink, visitors 1 to 500   : 50/50 {(arms_ab[:, :500] == 0).mean():.3f}, Thompson {(arms_ts[:, :500] == 0).mean():.3f}")
print(f"share shown pink, last 500 visitors   : 50/50 {(arms_ab[:, -500:] == 0).mean():.3f}, Thompson {(arms_ts[:, -500:] == 0).mean():.3f}")


**Reading.** The 50/50 test keeps sending half of the visitors to each page, and its regret grows in a straight line: 250 lost purchases after 5,000 visitors (half of the 5,000 visitors see black, and each loses 0.1 purchases on average). Thompson sampling starts at 50/50, since both posteriors are wide, then shifts: in the last 500 visitors pink gets 99.3% of the traffic. Its regret is 16, and the curve flattens. In this simulation the chance of showing black stays above zero at every visitor, because black's posterior is never certain, and it shrinks as the evidence grows. The curves are averages over 200 runs: an individual run can still go a long stretch without showing black, so they are not a guarantee about every run.

**Epsilon-greedy.** A simpler bandit rule keeps a running estimate $Q$ of each page's rate and updates it after each visitor with reward $r$ (1 for a buy, 0 for no buy):

$$ Q \leftarrow Q + \alpha\,(r - Q). $$

Here $\alpha$ is the step size. With probability $1-\varepsilon$ it shows the page with the larger $Q$ (exploit), and with probability $\varepsilon$ a random page (explore). Here $\varepsilon = 0.1$.

Plain facts about this update.

- It is an incremental estimate of an action's average immediate reward. It has the same "old estimate + step size × prediction error" structure used throughout reinforcement learning (RL). In Q-learning, the target also includes an estimate of the reward expected after the action. Here the target is just the observed reward, because this is a bandit.
- It is **stochastic gradient descent** on the squared error $\tfrac12 (r - Q)^2$ for a mean: the gradient with respect to $Q$ is $-(r-Q)$, so a step of size $\alpha$ is the update above. With $\alpha = 1/n$ ($n$ is the number of visitors shown that page so far) $Q$ is exactly the running average.

**A hand trace.** One page, $Q$ starts at 0, and the rewards are 1, 0, 1, with $\alpha = 1/n$.

| visitor $n$ | reward $r$ | $\alpha = 1/n$ | new $Q = Q + \alpha (r - Q)$ |
|---|---|---|---|
| 1 | 1 | 1 | $0 + 1 \times (1 - 0) = 1$ |
| 2 | 0 | 1/2 | $1 + \tfrac12 (0 - 1) = 0.5$ |
| 3 | 1 | 1/3 | $0.5 + \tfrac13 (1 - 0.5) = 0.6667$ |

The last value is the mean of 1, 0, 1. With a constant step such as $\alpha = 0.2$ the old rewards fade instead of being averaged equally. Exercise 9.5 (At home) asks you to write this update in a loop.


In [ ]:
arms_eg = run_policy("eps", seed=62)
print(f"regret after 5,000 visitors: 50/50 {regret_curve(arms_ab)[-1]:.0f}, Thompson {regret_curve(arms_ts)[-1]:.0f}, epsilon-greedy {regret_curve(arms_eg)[-1]:.0f}")
print(f"epsilon-greedy share shown pink in the last 500 visitors: {(arms_eg[:, -500:] == 0).mean():.3f}")

# check: with alpha = 1/n the update Q <- Q + alpha (r - Q) is the running mean
rewards = np.array([1, 0, 0, 1, 1, 0, 1])
Q_run = 0.0
for n_seen, r in enumerate(rewards, start=1):
    Q_run += (1 / n_seen) * (r - Q_run)
print("running update with alpha = 1/n:", round(Q_run, 4), " plain mean:", round(rewards.mean(), 4))


**Reading.** Epsilon-greedy loses 34 purchases. It sends 5% of traffic to the worse page forever (the random half of the 10% exploration goes to black half the time), so its regret keeps growing slowly. The last line confirms that $\alpha = 1/n$ gives the plain mean (0.5714 both ways).

**Check yourself.**

1. In this simulation, which method loses fewer purchases while learning? Which gives a simpler scientific answer, and why?
2. Why does Thompson sampling explore less over time, without being told to?

<details><summary>Answer</summary>

1. In this simulation, Thompson sampling loses fewer purchases while learning. Whether that holds elsewhere depends on the goal, the horizon, how late the rewards arrive, and the prior information. The fixed A/B design has a simple valid test of the difference. Adaptive allocation needs inference that accounts for the adaptive choices, so the ordinary fixed-design z-test cannot just be reused on the bandit's samples.
2. The posteriors narrow as counts grow. A narrow posterior for the worse page rarely produces a draw above the better page's draw.

</details>

This is explore and exploit, the core problem of the reinforcement learning course.


## 7. Importance sampling: judging a new rule with old data

The question that connects the night: the A/B test logged visitors under a 50/50 rule. Could we have known how a different rule, such as "always show pink", would have done, without running it?

A **policy** is a rule that gives a probability to each action (here, each page). The old policy showed pink with probability 0.5 and black with probability 0.5. A new policy might show pink with probability 1.

We evaluate a fixed rule with known action probabilities. We are not estimating the whole learning path of a Thompson sampler, which would have updated on different visitors and chosen different pages. (Doing that needs extra methods and is beyond this lab.)

The idea is to **reweight** the logged visitors. Each logged visitor gets the weight

$$ w = \frac{\text{chance the new rule shows this page}}{\text{chance the old rule showed this page}}. $$

Here is a tiny table for "always pink" under the old 50/50 rule (the first four rows are one possible log):

| page shown | reward $r$ | old probability | new probability | weight $w$ | $w \times r$ |
|---|---|---|---|---|---|
| pink | 1 | 0.5 | 1 | 2 | 2 |
| pink | 0 | 0.5 | 1 | 2 | 0 |
| black | 1 | 0.5 | 0 | 0 | 0 |
| black | 0 | 0.5 | 0 | 0 | 0 |

The estimate of the new rule's purchase rate is the average of $w \times r$ over all logged visitors, $r$ is 1 for a buy and 0 otherwise. The black rows have weight 0 but still count in the denominator (the number of visitors), because those visitors were logged. In the table the average is $(2 + 0 + 0 + 0)/4 = 0.5$.

**Support condition.** If the new rule might choose an action, the old rule must sometimes have chosen it. Otherwise the weight would divide by zero, and the log holds no information about that action.

The next cell makes a simulated log of 2,000 visitors under the 50/50 rule and applies the weights. We know the truth in this simulation: always pink earns 0.40.


In [ ]:
# Simulated log of an A/B test that showed pink or black with probability 0.5 each.
# True purchase rates (known only because this is a simulation): pink 0.40, black 0.30.
true_rates = np.array([0.40, 0.30])          # index 0 = pink, 1 = black
rng_log = np.random.default_rng(7)
n_log = 2000
log_page = rng_log.integers(0, 2, n_log)      # 0 = pink shown, 1 = black shown
log_buy = (rng_log.random(n_log) < true_rates[log_page]).astype(float)
print("visitors logged:", n_log, " shown pink:", int((log_page == 0).sum()), " shown black:", int((log_page == 1).sum()))
print("observed purchase rate, pink visitors :", round(log_buy[log_page == 0].mean(), 3))
print("observed purchase rate, black visitors:", round(log_buy[log_page == 1].mean(), 3))


In [ ]:
w_pink = np.where(log_page == 0, 1 / 0.5, 0.0)            # new rule: always pink, old rule: 50/50
est_always_pink = (w_pink * log_buy).mean()
print(f"importance-sampling estimate of 'always pink': {est_always_pink:.3f}   (truth: 0.400)")
print(f"naive average over all visitors (what a 50/50 mix earns): {log_buy.mean():.3f}")
print(f"average over pink visitors only: {log_buy[log_page == 0].mean():.3f}")


**Reading.** The estimate for "always pink" is 0.383, close to the truth 0.40. The naive average over all visitors, 0.344, answers a different question (what the 50/50 mix earns). With this rule only pink visitors count, each with weight 2, to make up for the black visitors we dropped. So the estimate is close to the average over pink visitors (0.387), and exactly equal to it only if exactly half of the visitors saw pink. Here 990 of the 2,000 did.

Exercise 9.4 below asks you to compute this estimate yourself.


### Exercise 9.4 (In class)

Task: estimate the purchase rate of the rule "always show pink" from the 50/50 log, by importance sampling. Give each logged visitor the weight (new chance) divided by (old chance), multiply by the purchase `log_buy`, and take the plain average over all `n_log` visitors (do not divide by the sum of weights).

Inputs: `log_page` (0 = pink shown, 1 = black shown), `log_buy` (0 or 1), from the simulated log above.

Define: `is_always_pink`, one number.


In [ ]:
# Exercise 9.4: define is_always_pink


In [ ]:
check("is_always_pink", is_always_pink, "0861eaeb0516a7ab")
print("ok  Exercise 9.4")


Next we check that it is right *on average*: repeat the whole experiment 1,000 times (simulated), and look at the spread of the estimates. We also try a second rule that shows pink 80% of the time, so its weights are $0.8/0.5 = 1.6$ for pink and $0.2/0.5 = 0.4$ for black. Its true rate is $0.8 \times 0.40 + 0.2 \times 0.30 = 0.38$.


In [ ]:
def repeat_is(pink_prob_old, pink_prob_new, n=2000, reps=1000, seed=71):
    """Simulated: reps logs of n visitors under the old rule; importance-sampling estimate of the new rule."""
    rng = np.random.default_rng(seed)
    page = (rng.random((reps, n)) >= pink_prob_old).astype(int)       # 0 = pink, 1 = black
    buy = (rng.random((reps, n)) < true_rates[page]).astype(float)
    new_prob = np.where(page == 0, pink_prob_new, 1 - pink_prob_new)
    old_prob = np.where(page == 0, pink_prob_old, 1 - pink_prob_old)
    return (new_prob / old_prob * buy).mean(axis=1)                    # one estimate per log

est_pink = repeat_is(0.5, 1.0)
est_80 = repeat_is(0.5, 0.8)
print(f"always pink, old rule 50/50 : mean {est_pink.mean():.3f}, sd {est_pink.std():.3f}   (truth 0.400)")
print(f"80% pink,    old rule 50/50 : mean {est_80.mean():.3f}, sd {est_80.std():.3f}   (truth 0.380)")


**Reading.** Over 1,000 repeats, the estimates average 0.401 for "always pink" (truth 0.400) and 0.381 for the 80% rule (truth 0.380). The weighted average is right on average: it is **unbiased**. The spread of one estimate is about 0.018 and 0.014.

**Predict.** Now suppose the old rule showed pink only 5% of the time. The weights for "always pink" become $1/0.05 = 20$. Will the estimate be right on average? Will its spread get smaller, the same, or larger? Write your answer, then run the cell.


In [ ]:
est_rare = repeat_is(0.05, 1.0, seed=72)
print(f"always pink, old rule shows pink 5% : mean {est_rare.mean():.3f}, sd {est_rare.std():.3f}   (truth 0.400)")
print(f"spread is {est_rare.std() / est_pink.std():.1f} times larger than with the 50/50 log")
print(f"smallest and largest estimate: {est_rare.min():.3f} and {est_rare.max():.3f}")

fig, ax = plt.subplots()
ax.hist(est_pink, bins=40, range=(0.1, 0.8), alpha=0.6, color="C0", label="old rule: pink 50%")
ax.hist(est_rare, bins=40, range=(0.1, 0.8), alpha=0.6, color="C3", label="old rule: pink 5%")
ax.axvline(0.40, color="black", linestyle="--")
ax.set_xlabel("estimate of the purchase rate of 'always pink'"); ax.set_ylabel("number of repeats"); ax.legend()
plt.show()


**Reading.** The estimate is still right on average (0.399), but one estimate is far less reliable: the spread is 0.063, about 3.5 times larger. Only about 100 of the 2,000 visitors saw pink, and each counts 20 times. A few lucky or unlucky buyers move the estimate a lot.

**Lesson:** you can only learn about a rule from data in which something sometimes did what the rule does. If the old rule almost never showed pink, the old data say little about "always pink".

**The idea in one line:** an average under one distribution, computed from samples drawn under another, by reweighting with the ratio of the two probabilities. In reinforcement learning this is **off-policy learning** (learning about one policy from the logs of another). It returns at home with Benter's $c$ and $q$.

**Check yourself.**

1. If the new rule showed black 100% of the time and the old rule showed black 50% of the time, what weight does a black visitor get?
2. Why are the weights from a rule that rarely did what the new rule does dangerous?
3. The old rule showed pink with probability 0 and the new rule is "always pink". What happens to the weight, and what can we learn?

<details><summary>Answer</summary>

1. $1/0.5 = 2$ for a black visitor, and 0 for a pink one.
2. The weights are huge (like 20), so a handful of logged visitors carry most of the estimate, and it becomes noisy.
3. The weight is $1/0 $, which is undefined: the support condition fails. The log has no pink visitors, so it cannot tell us anything about pink.

</details>


## 8. Why randomize: Washington's plots

George Washington tested crop rotations and fertilizers on his farm, and he divided his land into plots and gave each plot a treatment. He wrote down planting dates, weather, and yields. This is close to a modern experiment. A modern one adds the key step: the plot gets its treatment by chance.

Why does chance matter? Suppose a farmer puts the fertilizer on the better fields, because those are the ones he cares about. The fertilized fields then grow more, but we cannot tell how much comes from the fertilizer and how much from the better soil. The soil is a confounder, like the victim's race in Section 5. If the plots are chosen by coin flip, good and poor soil land in both groups about equally, and the difference between the groups is the effect of the fertilizer.

The SimCity test did this, if we assume (as such tests do) that the visitors were split at random: each visitor got a page by chance, so, on average, the other differences between visitors are balanced across the two groups (in one particular sample some chance imbalance remains). That is why the 10-point gap is read as the effect of the banner. 

The [Rent Control Lab](https://vsokolov.org/labs/homeless-lab.html) shows the other side: in an observational regression the estimate moves when you change which variables are in the model. Try three specifications and write down how the estimate changes.

**Check yourself.** In Section 5 the data were not randomized. Which step of a randomized experiment is missing there?

<details><summary>Answer</summary>

Assigning the group (the defendant's race, the victim's race) by chance. Without it, other differences between the groups, here the victim's race and the facts of the cases, can drive the outcome.

</details>


## 9. Russian election observers

In the Moscow elections to the State Duma on 4 December 2011, Enikolopov and coauthors (2013, *PNAS*) randomly sent independent observers to 156 of the 3,164 polling stations in Moscow. They compared the reported vote share of the ruling party, United Russia, at stations with and without observers. Because the stations were chosen by chance, other differences between stations are balanced between the groups on average (in one sample some chance imbalance remains). United Russia's share was about 11 percentage points lower where observers were present.

**The data below are simulated.** We use only the published summary: 3,164 stations, 156 with observers, a gap of about 11 points. We assume a control mean share of 0.46 and a spread across stations of 0.12; these two are made-up numbers for the simulation, not the paper's. The question is how the difference of means looks when labels are shuffled.


In [ ]:
rng_ru = np.random.default_rng(2011)
n_stations, n_obs = 3164, 156
has_observer = np.zeros(n_stations, dtype=bool)
has_observer[rng_ru.choice(n_stations, n_obs, replace=False)] = True        # assigned at random
share = np.clip(0.46 - 0.11 * has_observer + rng_ru.normal(0, 0.12, n_stations), 0, 1)   # simulated vote share

diff_obs = share[has_observer].mean() - share[~has_observer].mean()
se_ru = np.sqrt(share[has_observer].var(ddof=1) / n_obs + share[~has_observer].var(ddof=1) / (n_stations - n_obs))
print(f"simulated mean share, with observers {share[has_observer].mean():.3f}, without {share[~has_observer].mean():.3f}")
print(f"difference = {diff_obs:.3f}, standard error = {se_ru:.4f}, 95% interval ({diff_obs - 1.96 * se_ru:.3f}, {diff_obs + 1.96 * se_ru:.3f})")


**Reading.** In the simulation, the stations with observers have a mean share of 0.340 and the others 0.461. The difference is -0.121, with a 95% interval of (-0.139, -0.103). The made-up true gap in the simulation is -0.11, and the simulated gap is -0.121. The difference is chance (a different random draw of noise), and -0.11 lies inside the printed interval. The interval shows how precisely 156 stations can measure the gap.

The [Rent Control Lab](https://vsokolov.org/labs/homeless-lab.html) also has a permutation test: run it and compare how its shuffled differences look with ours below.

**Your turn: a permutation test.** The null hypothesis says the observer made no difference, so the labels "observer" and "no observer" could be shuffled freely. Shuffle the labels many times, compute the difference of means each time, and see where the real difference falls. The cell runs with 2,000 shuffles. Target: the real difference falls far outside the shuffled ones, so no shuffle is as extreme. (The p-value is then reported as (count + 1)/(n_perm + 1), never as exactly 0.) Change `n_perm` and look at the spread of the shuffled differences.


In [ ]:
n_perm = 2000   # change this
rng_pm = np.random.default_rng(9)
diffs = np.empty(n_perm)
for i in range(n_perm):
    shuffled = rng_pm.permutation(has_observer)                 # same number of observer stations, random ones
    diffs[i] = share[shuffled].mean() - share[~shuffled].mean()
n_extreme = int((np.abs(diffs) >= abs(diff_obs)).sum())     # np.abs: two-sided, a gap of either sign counts
p_perm = (n_extreme + 1) / (n_perm + 1)                       # never report exactly 0 from a finite number of shuffles
print(f"shuffled differences: sd {diffs.std():.4f}, range ({diffs.min():.3f}, {diffs.max():.3f})")
print(f"real difference {diff_obs:.3f}; shuffles at least as extreme: {n_extreme} of {n_perm}")
print(f"permutation p-value (count + 1)/(n_perm + 1) = {p_perm:.4f}")


**Reading.** The shuffled differences have standard deviation 0.0101 and stay within about 0.04 of zero. The real difference, -0.121, is far outside, in none of the 2000 shuffles. So the p-value estimate is (0 + 1)/(2000 + 1) = 0.0005. We use `np.abs` so the test is two-sided: a shuffled gap of either sign counts as extreme. This is the randomization logic: if the observer did nothing, a gap this large would almost never come from the random choice of stations alone.

The Gujarat pollution-audit experiment (Duflo and coauthors, 2013) is the next section.


## 10. Gujarat pollution audits (simulated)

Duflo and coauthors (2013) ran a randomized experiment in Gujarat, India. Plants paid their own environmental auditors, and auditors tended to report pollution just under the legal limit even when the true value was higher. In the treatment group, auditors were assigned at random, paid from a central pool, and checked by random backchecks. A **backcheck** is an independent repeat measurement of the same plant by a technical agency, used to catch false reports.

This is the same story as the lemons problem (Lab 8: when one side knows more and the other cannot check, the market breaks): when the seller pays the inspector, the inspection says what the seller wants.

**The data below are simulated**, not from the paper: true emissions of 400 plants (median about 130 against a legal limit of 150). In the control group, a report is moved to just under the limit, with probability `p_fudge`, when the truth is above the limit. In the treatment group, reports are honest. We look for bunching: a pile-up of reports just below 150.


In [ ]:
rng_g = np.random.default_rng(94)
limit = 150
truth = rng_g.lognormal(mean=np.log(130), sigma=0.35, size=400)
p_fudge = 0.8   # change this
fudge = (truth > limit) & (rng_g.random(400) < p_fudge)
report_control = np.where(fudge, rng_g.uniform(0.85 * limit, limit - 0.5, 400), truth)    # auditor paid by the plant
report_treat = truth.copy()                                                              # honest reports

band = lambda x: ((x >= 0.75 * limit) & (x < limit)).mean()
print(f"share of reports between 75% and 100% of the limit: control {band(report_control):.3f}, treatment {band(report_treat):.3f}")
print(f"share of reports above the limit: control {(report_control >= limit).mean():.3f}, treatment {(report_treat >= limit).mean():.3f}")
bins = np.arange(40, 360, 10)
plt.hist(report_treat, bins=bins, alpha=0.6, color="C0", label="treatment (simulated)")
plt.hist(report_control, bins=bins, alpha=0.6, color="C3", label="control (simulated)")
plt.axvline(limit, color="black", linestyle="--")
plt.xlabel("reported pollution (mg per cubic meter)"); plt.ylabel("number of plants"); plt.legend()
plt.show()


**Reading.** The control reports bunch just below the limit: 0.645 of control reports lie between 75% and 100% of the limit, against 0.365 in the treatment group. Reports above the limit are 0.080 for control and 0.360 for treatment, although the true emissions are the same in both. Treatment shows no extra pile-up just below 150. The bunching is the sign that the reports, not the plants, were changed. Your turn: change `p_fudge` (the line marked `# change this`) and watch the bunching grow or shrink.


## Exit problem (paper, no code)

Do this on paper, as on the midterm. Show the formula and the numbers.

(a) A page starts with a Beta(1, 1) prior. In 40 visitors, 12 buy. Write the posterior and its mean.

(b) Page A: 45 buys of 100 visitors. Page B: 36 buys of 100 visitors. Compute the difference of the two rates and its 95% confidence interval (unpooled standard errors, $z = 1.96$). Does the interval contain 0?


## At home

Two graded exercises: 9.5 (the bandit update in a loop) and 9.6 (Benter's importance sampling). Make sure Exercises 9.1 to 9.4 are filled in too.


In [ ]:
# Simulated log for the bandit update: 40 visitors, arm 0 = pink, arm 1 = black.
# Pink's true rate is 0.6 for the first 20 visitors and 0.2 after that; black stays at 0.4.
rng_q = np.random.default_rng(55)
T_q = 40
arm_log = rng_q.integers(0, 2, T_q)
rate_now = np.where(np.arange(T_q) < 20, 0.6, 0.2)
reward_log = (rng_q.random(T_q) < np.where(arm_log == 0, rate_now, 0.4)).astype(float)
print("visitors:", T_q, " shown pink:", int((arm_log == 0).sum()), " shown black:", int((arm_log == 1).sum()))
print("plain mean reward, pink :", round(reward_log[arm_log == 0].mean(), 3), " black:", round(reward_log[arm_log == 1].mean(), 3))


### Exercise 9.5 (At home)

Task: write the bandit update $Q \leftarrow Q + \alpha\,(r - Q)$ in a loop over the logged visitors, and compare two step sizes. Keep one estimate per page, both starting at 0. The first estimate uses $\alpha = 1/n$, where $n$ counts the visitors shown that page so far (including this one). The second uses a constant $\alpha = 0.2$. Inside the loop the key line is `Q[a] += alpha * (r - Q[a])`, with `a` the page shown and `r` the reward.

Inputs: `arm_log` (0 = pink, 1 = black) and `reward_log` (0 or 1), 40 simulated visitors, from the cell above.

Define: `q_1n` and `q_const`, two arrays of shape (2,) holding the final estimates for (pink, black). Afterwards look at the plain means printed above: which step size is closer to the recent behavior of pink, whose true rate dropped at visitor 20?


In [ ]:
# Exercise 9.5: define q_1n, q_const


In [ ]:
check_array("q_1n", q_1n, "ad8383596032fb44", (2,))
check_array("q_const", q_const, "51f37c1ab755c5cd", (2,))
print("ok  Exercise 9.5")


In [ ]:
# Race 1159 (Happy Valley, 25 November 1998): final odds for the ten horses (from the Lab 8 notebook)
odds = np.array([8.1, 9.5, 3.5, 19.0, 9.7, 13.0, 6.0, 6.2, 14.0, 13.0])
q = (1 / odds) / (1 / odds).sum()            # the crowd's chances
c = q**1.06 / (q**1.06).sum()                # Benter's chances (stretched crowd chances)
h = 2                                         # index of horse 3 (counting from 0)
print("horse 3: odds", odds[h], " crowd chance q =", round(q[h], 4), " Benter chance c =", round(c[h], 4))


### Exercise 9.6 (At home)

Task: Benter's importance sampling. In race 1159, the winner is drawn from the crowd's chances $q$. Estimate the expected profit of a \$1 bet on horse 3 under Benter's chances $c$ (not the crowd's) from 100,000 simulated races: in each race, the profit is (odds $-$ 1) if horse 3 wins and $-1$ if it loses. Weight each race by $c/q$ of the *winner*, and average, which means: estimate $= \text{mean}\big(\text{profit} \times c_{\text{winner}}/q_{\text{winner}}\big)$. Draw the winners with your own `np.random.default_rng(seed)`. A \$1 winning ticket on odds $O$ pays $O$ in all, so the profit on a win is $O - 1$.

Inputs: `odds`, `q`, `c`, `h` (index of horse 3) from the cell above.

Define: `benter_profit`, a number. The check allows a tolerance. The exact answer is the sum over horses: $c_h \cdot \text{odds}_h - 1$, printed after the check.


In [ ]:
# Exercise 9.6: define benter_profit


In [ ]:
assert benter_profit is not None, "replace None for benter_profit"
assert abs(benter_profit - -0.146699) < 0.03, "benter_profit is not within 0.03 of the target"
print("ok  Exercise 9.6")


In [ ]:
print("exact expected profit under c:", round(c[h] * odds[h] - 1, 4))
print("expected profit under q (the crowd):", round(q[h] * odds[h] - 1, 4))


## Submitting

Before you submit, choose Kernel, then Restart and Run All, and confirm that every check cell prints `ok`. Submit the `.ipynb` file on Canvas before the next class.
